<a href="https://colab.research.google.com/github/akashkguw/orion/blob/main/orion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Orion: Sparse Attention for Long-Context Transformers

This notebook demonstrates training decoder-only Transformers with sparse attention (window + expander edges) for efficient long-context modeling.

**Key Features:**
- Sparse Attention - O(T*(W+d)) complexity vs O(T^2) for dense
- Reproducible - Deterministic training with seed control
- Configurable - YAML-based configs for easy experimentation
- Well-tested - 160+ tests covering all components


## Setup: Install Dependencies

In [17]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = True
REPO_URL = "https://github.com/akashkguw/orion.git"
REPO_REF = os.environ.get("ORION_GIT_REF", "main")  # set to branch/tag/sha if needed

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/orion")
    REPO_ROOT = DRIVE_ROOT / "repo"
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
else:
    REPO_ROOT = Path("/content/orion") if IN_COLAB else Path.cwd()


def _run(cmd: list[str], *, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    proc = subprocess.run(cmd, cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"Command failed ({' '.join(cmd)}):\nSTDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    return proc


def _is_git_repo(path: Path) -> bool:
    return (path / ".git").exists()


def _clone_repo(repo_url: str, repo_root: Path, retries: int = 3) -> None:
    last_error = "unknown"
    for attempt in range(1, retries + 1):
        proc = subprocess.run(
            ["git", "clone", repo_url, str(repo_root)],
            text=True,
            capture_output=True,
        )
        if proc.returncode == 0:
            return

        last_error = (proc.stderr or proc.stdout or "clone failed").strip()
        tail = last_error.splitlines()[-1] if last_error else "clone failed"
        print(f"Clone attempt {attempt}/{retries} failed: {tail}")

        if repo_root.exists() and not _is_git_repo(repo_root):
            shutil.rmtree(repo_root, ignore_errors=True)

        if attempt < retries:
            time.sleep(attempt)

    raise RuntimeError(f"Failed to clone repository after {retries} attempts: {last_error}")


def _sync_repo(repo_root: Path, ref: str) -> None:
    # Fail fast on dirty trees so runs stay reproducible.
    status = _run(["git", "status", "--porcelain"], cwd=repo_root)
    if status.stdout.strip():
        raise RuntimeError(
            "Repository has local changes in Drive clone. "
            "Commit/stash/remove them, or delete the repo folder and rerun."
        )

    _run(["git", "fetch", "--all", "--tags", "--prune"], cwd=repo_root)

    # Checkout requested ref; works for local branch, remote branch, tag, or sha.
    checkout = _run(["git", "checkout", ref], cwd=repo_root, check=False)
    if checkout.returncode != 0:
        # Try remote branch fallback.
        _run(["git", "checkout", "-B", ref, f"origin/{ref}"], cwd=repo_root)

    # Fast-forward if branch tracks remote.
    branch = _run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=repo_root).stdout.strip()
    if branch != "HEAD":
        _run(["git", "pull", "--ff-only", "origin", branch], cwd=repo_root)


if REPO_ROOT.exists() and not _is_git_repo(REPO_ROOT):
    if any(REPO_ROOT.iterdir()):
        backup = REPO_ROOT.with_name(f"{REPO_ROOT.name}_backup_{int(time.time())}")
        print(f"Found non-git directory at {REPO_ROOT}; moving to {backup}")
        REPO_ROOT.rename(backup)
    else:
        REPO_ROOT.rmdir()

if not REPO_ROOT.exists():
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    print(f"Cloning {REPO_URL} -> {REPO_ROOT}")
    _clone_repo(REPO_URL, REPO_ROOT)
else:
    print(f"Using existing repository at {REPO_ROOT}")

_sync_repo(REPO_ROOT, REPO_REF)
os.chdir(REPO_ROOT)

commit = _run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT).stdout.strip()
branch = _run(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=REPO_ROOT).stdout.strip()

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")
print(f"Checked out: {branch} @ {commit}")
if IN_COLAB and USE_GOOGLE_DRIVE:
    print(f"Google Drive persistence enabled at: {REPO_ROOT}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning https://github.com/akashkguw/orion.git -> /content/drive/MyDrive/orion/repo
IN_COLAB=True
REPO_ROOT=/content/drive/MyDrive/orion/repo
Checked out: main @ 7d2dc83
Google Drive persistence enabled at: /content/drive/MyDrive/orion/repo


In [18]:
%cd {REPO_ROOT}
!ls
!pip -q install -r requirements.txt -r requirements-dev.txt

/content
COMPREHENSIVE_METRICS_GUIDE.md	README.md
configs				requirements-dev.txt
experiment.ipynb		requirements.txt
LICENSE				SPARSE_ATTENTION_ARCHITECTURE.md
Makefile			sparse_hyperparameter_tuning.ipynb
orion				sparse_norm_control_tuning.ipynb
orion.ipynb			tests
pyproject.toml


In [19]:
import importlib.util
from pathlib import Path

print("torch installed:", importlib.util.find_spec("torch") is not None)

sparse_py = Path("orion/attention/sparse.py")
text = sparse_py.read_text(encoding="utf-8")
required = "No eager fallback is allowed in strict flex mode."
legacy = "retrying with eager flex_attention."

if required not in text:
    raise RuntimeError(
        "Stale sparse.py detected: strict-flex guard missing. Sync repo and restart runtime."
    )
if legacy in text:
    raise RuntimeError(
        "Legacy eager-fallback warning string found. Repo is not on the expected revision."
    )

print("Sparse flex guard verified in source.")



torch installed: True
Sparse flex guard verified in source.


In [20]:
# If Torch is not available, uncomment and run:
# !pip -q install torch

## What is Sparse Attention?

Sparse attention reduces computational complexity from O(T^2) to O(T*(W+d)) by attending to only a subset of positions:

- **Local Window** - Dense context for nearby tokens (short-range dependencies)
- **Expander Edges** - Structured long-range connections using modular arithmetic

### Example Pattern

For query at position q=10 with window_size=4, expander_degree=3:

```
Window:   [7, 8, 9, 10]           (local context)
Expander: [9, 6, 1]               (long-range via quadratic residues)
Union:    [1, 6, 7, 8, 9, 10]     (7 positions vs 11 for dense)
```

### Performance

| Metric | Dense | Sparse | Speedup |
|--------|-------|--------|----------|
| Complexity | O(T^2) | O(T*(W+d)) | ~7x |
| Memory | O(T^2*Dh) | O(T*(W+d)*Dh) | ~7x |
| Example (T=512, W=64, d=8) | 262K pos/query | 36.8K pos/query | 7.1x |

## Quick Start: Train with Dense Attention

### Execute Training (Smoke Test - 20 steps)

In [21]:
!python -m orion.train --config configs/golden.yaml

Step 1: loss=5.7179, ppl=304.26, throughput=2479.5 tok/s, grad_norm=0.3906, clipped=False, step_time=412.98ms, acc=0.0029, lr=3.00e-04
Step 2: loss=5.6898, ppl=295.84, throughput=93107.7 tok/s, grad_norm=0.3939, clipped=False, step_time=11.00ms, acc=0.0078, lr=3.00e-04
Step 3: loss=5.7150, ppl=303.39, throughput=117541.5 tok/s, grad_norm=0.3887, clipped=False, step_time=8.71ms, acc=0.0049, lr=3.00e-04
Step 4: loss=5.6910, ppl=296.18, throughput=114912.4 tok/s, grad_norm=0.3959, clipped=False, step_time=8.91ms, acc=0.0029, lr=3.00e-04
Step 5: loss=5.7270, ppl=307.04, throughput=123012.1 tok/s, grad_norm=0.3959, clipped=False, step_time=8.32ms, acc=0.0049, lr=3.00e-04
Step 6: loss=5.7056, ppl=300.56, throughput=106919.8 tok/s, grad_norm=0.3930, clipped=False, step_time=9.58ms, acc=0.0029, lr=3.00e-04
Step 7: loss=5.6942, ppl=297.13, throughput=94016.8 tok/s, grad_norm=0.3962, clipped=False, step_time=10.89ms, acc=0.0059, lr=3.00e-04
Step 8: loss=5.7138, ppl=303.02, throughput=120757.1 to

### Evaluate Model

In [22]:
!python -m orion.eval --config configs/golden.yaml --checkpoint runs/latest/checkpoint.pt

{'loss': 5.670691013336182, 'ppl': 290.2350158691406}


### View Training Logs

In [23]:
!tail -n 5 runs/latest/metrics.jsonl

{"type": "step", "step": 16, "loss": 5.6980156898498535, "ppl": 298.27494335400047, "throughput_tokens_per_sec": 102632.55821066718, "grad_norm": 0.3877656161785126, "grad_clipped": false, "diverged": false, "loss_spike": false, "grad_norm_spike": false, "step_time_ms": 9.977340698242188, "accuracy_top1": 0.0048828125, "learning_rate": 0.0003, "wall_time_s": 0.6373779773712158}
{"type": "step", "step": 17, "loss": 5.688562393188477, "ppl": 295.468547573142, "throughput_tokens_per_sec": 103070.96942644588, "grad_norm": 0.38964077830314636, "grad_clipped": false, "diverged": false, "loss_spike": false, "grad_norm_spike": false, "step_time_ms": 9.93490219116211, "accuracy_top1": 0.005859375, "learning_rate": 0.0003, "wall_time_s": 0.650780200958252}
{"type": "step", "step": 18, "loss": 5.720730304718018, "ppl": 305.12767779024324, "throughput_tokens_per_sec": 122485.87754170825, "grad_norm": 0.3912482261657715, "grad_clipped": false, "diverged": false, "loss_spike": false, "grad_norm_spik

## Train with Sparse Attention

### Sparse Attention Configuration

Sparse attention is configured via YAML:

```yaml
attention:
  backend: sparse
  window_size: 64
  expander_degree: 16
  sparse_impl: flex
  sparse_block_size: 64
  sparse_probe_every: 0    # set >0 only when you need probe diagnostics
  sparse_probe_tokens: 256 # optional: probe length cap
```

**Notes:**
- With `sparse_impl: flex`, attention-weight metrics can show `NA` unless probe metrics are enabled.
- `sparse_probe_every: 0` disables probe metrics (lowest overhead).

**When to use sparse attention:**
- Long sequences (512+)
- Limited GPU memory
- Need stable training with diverse patterns
- Short sequences (<256) - dense is often faster


### Window Attention (Sliding Local Context)

Window attention is a causal local-attention baseline between dense and sparse:

- Each token attends only to the last `W` tokens (including itself).
- Complexity is `O(T*W)` instead of dense `O(T^2)`.
- No expander edges: simpler and usually more stable than sparse.

Configuration example:

```yaml
attention:
  backend: window
  window_size: 256
```

Use when:
- you want a strong local-context baseline,
- sparse expander behavior is not required,
- you want lower memory than dense without sparse-index complexity.


### Sparse Attention Smoke Test (5 steps)

In [24]:
!python -m orion.train --config configs/exp_orion_sparse_smoke.yaml

Step 1: loss=5.7202, ppl=304.97, throughput=207.4 tok/s, grad_norm=0.3900, clipped=False, step_time=4936.39ms, acc=0.0020, lr=3.00e-04
Step 2: loss=5.7016, ppl=299.33, throughput=37321.9 tok/s, grad_norm=0.3938, clipped=False, step_time=27.44ms, acc=0.0029, lr=3.00e-04
Step 3: loss=5.7016, ppl=299.35, throughput=50372.6 tok/s, grad_norm=0.3897, clipped=False, step_time=20.33ms, acc=0.0000, lr=3.00e-04
Step 4: loss=5.6982, ppl=298.34, throughput=52782.5 tok/s, grad_norm=0.3950, clipped=False, step_time=19.40ms, acc=0.0098, lr=3.00e-04
Step 5: loss=5.7216, ppl=305.39, throughput=56303.8 tok/s, grad_norm=0.3976, clipped=False, step_time=18.19ms, acc=0.0020, lr=3.00e-04


### Evaluate Sparse Attention Model

In [25]:
!python -m orion.eval --config configs/exp_orion_sparse_smoke.yaml --checkpoint runs/exp_orion_sparse_smoke/checkpoint.pt

{'loss': 5.699633598327637, 'ppl': 298.7579040527344}


## Quick A/B Runs: Dense Wins vs Sparse Wins


### Two Fast Comparison Presets (<=1000 steps)

This section creates two paired Dense vs Sparse runs with the same model, seed, and LR in each pair:

1. Run A (`seq_len=256`, `batch_size=16`, `steps=600`)
Expected winner: **Dense** on throughput/step-time.
2. Run B (`seq_len=2048`, `batch_size=1`, `steps=600`)
Expected winner: **Sparse (flex)** on memory/scalability, and often throughput at long context.

Notes:
- Sparse is **flex-only** in this notebook (`sparse_impl: flex`), no gather path.
- `sparse_probe_every=0` keeps quick runs low overhead.
- Keep steps <=1000 for quick Colab iteration.


In [26]:
from pathlib import Path

import yaml

STEPS_SHORT = 600
STEPS_LONG = 600
LR = 3e-4
SEED = 123

MODEL = {
    "name": "orion",
    "d_model": 256,
    "n_layers": 4,
    "n_heads": 4,
    "mlp_mult": 4,
}

def build_common(*, seq_len: int, batch_size: int, steps: int):
    return {
        "run": {
            "seed": SEED,
            "steps": steps,
            "log_every": 50,
            "save_every": steps,
        },
        "data": {
            "dataset": "tinyshakespeare",
            "root": "data",
            "seq_len": seq_len,
            "batch_size": batch_size,
        },
        "model": dict(MODEL),
        "optim": {"lr": LR},
        "eval": {"long_context_batch_size": 1},
    }

def dense_cfg(*, seq_len: int, batch_size: int, steps: int, out_dir: str):
    base = build_common(seq_len=seq_len, batch_size=batch_size, steps=steps)
    return {
        **base,
        "run": {**base["run"], "out_dir": out_dir},
        "attention": {"backend": "dense"},
    }

def sparse_cfg(
    *,
    seq_len: int,
    batch_size: int,
    steps: int,
    out_dir: str,
    window_size: int,
    expander_degree: int,
):
    base = build_common(seq_len=seq_len, batch_size=batch_size, steps=steps)
    return {
        **base,
        "run": {**base["run"], "out_dir": out_dir},
        "attention": {
            "backend": "sparse",
            "window_size": window_size,
            "expander_degree": expander_degree,
            "sparse_impl": "flex",
            "sparse_block_size": 64,
            "sparse_probe_every": 0,
            "sparse_probe_tokens": 256,
        },
    }

cfg_specs = [
    (
        "quick_dense_win_dense_t256.yaml",
        dense_cfg(
            seq_len=256,
            batch_size=16,
            steps=STEPS_SHORT,
            out_dir="runs/quick_dense_win_dense_t256",
        ),
    ),
    (
        "quick_dense_win_sparse_t256_w64_d8.yaml",
        sparse_cfg(
            seq_len=256,
            batch_size=16,
            steps=STEPS_SHORT,
            out_dir="runs/quick_dense_win_sparse_t256_w64_d8",
            window_size=64,
            expander_degree=8,
        ),
    ),
    (
        "quick_sparse_win_dense_t2048.yaml",
        dense_cfg(
            seq_len=2048,
            batch_size=1,
            steps=STEPS_LONG,
            out_dir="runs/quick_sparse_win_dense_t2048",
        ),
    ),
    (
        "quick_sparse_win_sparse_t2048_w64_d8.yaml",
        sparse_cfg(
            seq_len=2048,
            batch_size=1,
            steps=STEPS_LONG,
            out_dir="runs/quick_sparse_win_sparse_t2048_w64_d8",
            window_size=64,
            expander_degree=8,
        ),
    ),
]

cfg_dir = Path("configs")
cfg_dir.mkdir(parents=True, exist_ok=True)

for filename, cfg in cfg_specs:
    path = cfg_dir / filename
    path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    print(f"Wrote {path}")


Wrote configs/quick_dense_win_dense_t256.yaml
Wrote configs/quick_dense_win_sparse_t256_w64_d8.yaml
Wrote configs/quick_sparse_win_dense_t2048.yaml
Wrote configs/quick_sparse_win_sparse_t2048_w64_d8.yaml


### Run A: Short Context (`T=256`) - Dense Should Win

Run dense first, then sparse with matching data/model/steps.


In [27]:
!python -m orion.train --config configs/quick_dense_win_dense_t256.yaml
!python -m orion.train --config configs/quick_dense_win_sparse_t256_w64_d8.yaml


Step 50: loss=2.6075, ppl=13.57, throughput=95303.9 tok/s, grad_norm=0.4087, clipped=False, step_time=42.98ms, acc=0.2515, lr=3.00e-04
  Window 50: vram=504MB, div_rate=0.000, act_norm=1.0092, attn_ent=4.3466 (norm=0.7839), clip_rate=0.140, attn_score_mean=0.4432
Step 100: loss=2.5350, ppl=12.62, throughput=97077.9 tok/s, grad_norm=0.4706, clipped=False, step_time=42.19ms, acc=0.2573, lr=3.00e-04
  Window 100: vram=507MB, div_rate=0.000, act_norm=1.0159, attn_ent=4.2550 (norm=0.7673), clip_rate=0.000, attn_score_mean=0.5101
Step 150: loss=2.4890, ppl=12.05, throughput=97329.8 tok/s, grad_norm=0.3926, clipped=False, step_time=42.08ms, acc=0.2703, lr=3.00e-04
  Window 150: vram=507MB, div_rate=0.000, act_norm=1.0210, attn_ent=4.2273 (norm=0.7623), clip_rate=0.000, attn_score_mean=0.5447
Step 200: loss=2.5184, ppl=12.41, throughput=100026.6 tok/s, grad_norm=0.5224, clipped=False, step_time=40.95ms, acc=0.2715, lr=3.00e-04
  Window 200: vram=507MB, div_rate=0.000, act_norm=1.0249, attn_ent

### Run B: Long Context (`T=2048`) - Sparse Should Win

This pair targets long-context scaling. Sparse should be better on memory/scalability,
and often throughput as context grows.


In [28]:
!python -m orion.train --config configs/quick_sparse_win_dense_t2048.yaml
!python -m orion.train --config configs/quick_sparse_win_sparse_t2048_w64_d8.yaml


Step 50: loss=2.6939, ppl=14.79, throughput=26787.1 tok/s, grad_norm=0.5831, clipped=False, step_time=76.45ms, acc=0.2344, lr=3.00e-04
  Window 50: vram=978MB, div_rate=0.000, act_norm=1.0080, attn_ent=6.3771 (norm=0.8364), clip_rate=0.140, attn_score_mean=0.4698
Step 100: loss=2.6411, ppl=14.03, throughput=27346.8 tok/s, grad_norm=0.5975, clipped=False, step_time=74.89ms, acc=0.2451, lr=3.00e-04
  Window 100: vram=980MB, div_rate=0.000, act_norm=1.0129, attn_ent=6.2052 (norm=0.8138), clip_rate=0.000, attn_score_mean=0.5531
Step 150: loss=2.5475, ppl=12.77, throughput=27601.0 tok/s, grad_norm=0.5467, clipped=False, step_time=74.20ms, acc=0.2427, lr=3.00e-04
  Window 150: vram=980MB, div_rate=0.000, act_norm=1.0167, attn_ent=6.1101 (norm=0.8014), clip_rate=0.000, attn_score_mean=0.6244
Step 200: loss=2.4781, ppl=11.92, throughput=27473.5 tok/s, grad_norm=0.5284, clipped=False, step_time=74.54ms, acc=0.2847, lr=3.00e-04
  Window 200: vram=980MB, div_rate=0.000, act_norm=1.0198, attn_ent=

## Comprehensive Metrics Logging

Orion logs comprehensive P0 metrics to `runs/latest/metrics.jsonl` in JSONL format:

### Step Metrics (every step)
- `loss`, `ppl` - Training loss and perplexity
- `throughput_tokens_per_sec` - Training throughput
- `grad_norm`, `grad_clipped` - Gradient statistics
- `step_time_ms` - Time per step
- `accuracy_top1` - Next-token prediction accuracy
- `learning_rate` - Current learning rate

### Window Metrics (every 50 steps)
- `vram_peak_mib` - Peak GPU memory usage
- `divergence_rate` - Fraction of diverged steps
- `activation_norm_rms` - RMS of residual stream
- `attention_entropy`, `attention_entropy_normalized` - Attention distribution entropy
- `clip_rate` - Fraction of clipped gradient steps
- `valid_neighbor_fraction` - (Sparse only) Effective degree / degree
- `attention_mass_window_pct`, `attention_mass_expander_pct` - (Sparse only) Mass split

### Run Metrics (once per run)
- `attention_degree` - Total attention neighbors (W + d)
- `compute_proxy_per_token`, `compute_proxy_per_seq`, `compute_proxy_per_step` - Complexity estimates

### Eval Metrics (every 1000 steps)
- `eval_ppl_512`, `eval_ppl_1024`, `eval_ppl_2048`, `eval_ppl_4096` - Long-context evaluation

**Important:** for fused sparse (`sparse_impl: flex`), weight-derived metrics may be `NA` unless probe metrics are enabled.

View metrics with:
```bash
cat runs/latest/metrics.jsonl | jq .
```


### Create Custom Attention Config

You can create custom configs by modifying YAML files.

Sparse example (performance-focused):

```yaml
attention:
  backend: sparse
  window_size: 64
  expander_degree: 16
  sparse_impl: flex
  sparse_block_size: 64
  sparse_probe_every: 0   # disable probe for clean perf benchmarking
  sparse_probe_tokens: 256
```

Sparse example (debug/analysis-focused):

```yaml
attention:
  backend: sparse
  window_size: 64
  expander_degree: 16
  sparse_impl: flex
  sparse_block_size: 64
  sparse_probe_every: 50  # enable periodic probe for entropy/mass diagnostics
  sparse_probe_tokens: 256
```

Window example:

```yaml
attention:
  backend: window
  window_size: 256
```



## Resources and Documentation

- README: Full documentation with examples
- SPARSE_ATTENTION_ARCHITECTURE.md: Deep technical dive into sparse attention
- GitHub: https://github.com/akashkguw/orion

### Key Files

- orion/attention/sparse.py - Sparse attention implementation
- orion/attention/dense.py - Dense attention implementation
- configs/ - Training configurations
- tests/test_sparse_attention.py - 82+ tests for sparse attention

## Troubleshooting

### Out of Memory (OOM)
- Use sparse attention instead of dense
- Reduce batch_size in config
- Reduce seq_len in config
- Reduce d_model in config

### Slow Training
- Check GPU is being used: !nvidia-smi
- Use sparse attention for long sequences
- Increase batch_size if memory allows

### Training Not Converging
- Try different seed values
- Adjust learning rate (lr in config)
- Increase training steps (steps in config)
- Check data loading with !head -c 100 data/tinyshakespeare.txt